# Intrinsic Benchmarks - Information Density

This notebook computes and visualizes the information density - intrinsic benchmarking metrics for the connectivity matrices of each of the PySPI and skarf methods.

Averages runs per subject first via `load_avg_mats_and_impose_sparsity`,
then computes all metrics on the averaged matrix.

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl

PROJECT_ROOT = Path("/home/jpillai/projects/skarf-experiments")
# sys.path.insert(0, str(PROJECT_ROOT / "src"))

from arfcexp.matrices import load_avg_mats_and_impose_sparsity, load_symmetry_lookup
from arfcexp.info_density import compute_all as compute_all_density

# paths
PARQUET_PATH = Path("/srv/projects/skarf/data_aggregation/hcp_1200_rfmri_schaefer.parquet")
SUBJECT_LIST = PROJECT_ROOT / "resources/subject_lists/hcp_complete_data_867_subject_list.txt"
OUT_DIR      = PROJECT_ROOT / "results/intrinsic_benchmarks"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# run params
SPARSITY=0.8
# N_JOBS=1          
TSP_NODES=list(range(50))
SMALL_WORLD_KWARGS=dict(seed=70, nrand=1, nswap=2000)
RICH_CLUB_KWARGS={"normalized": False}

# subjects
with open(SUBJECT_LIST) as f:
    SUB_LIST = [line.strip() for line in f if line.strip()]

SYMMETRY_LOOKUP=load_symmetry_lookup(PROJECT_ROOT)

print(f"Subjects : {len(SUB_LIST)}")
print(f"Output   : {OUT_DIR}")

Subjects : 867
Output   : /home/jpillai/projects/skarf-experiments/results/intrinsic_benchmarks


In [2]:
schema = pl.scan_parquet(PARQUET_PATH).schema
has_lag = "lag" in schema

select_cols = ["method", "func"] + (["lag"] if has_lag else [])
combos = (
    pl.scan_parquet(PARQUET_PATH)
    .filter(pl.col("success"))
    .select(select_cols)
    .unique()
    .collect()
    .to_pandas()
)
if not has_lag:
    combos["lag"] = None

combos = combos.reset_index(drop=True)
print(f"{len(combos)} (method, func, lag) combinations found")
combos

173 (method, func, lag) combinations found


/tmp/ipykernel_898546/2597384383.py:1: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  schema = pl.scan_parquet(PARQUET_PATH).schema


,method,func,lag
0,pyspi,te_kernel_W-0.25_k-1,NaN
1,pyspi,pdist_braycurtis,NaN
2,skarf,linear_lasso,1.0
3,pyspi,phase_multitaper_max_fs-1_fmin-0-25_fmax-0-5,NaN
4,pyspi,prec_GraphicalLassoCV,NaN
...,...,...,...
168,skarf,linear_enet-pos,0.0
169,pyspi,psi_wavelet_mean_fs-1_fmin-0_fmax-0-25_mean,NaN
170,pyspi,icoh_multitaper_mean_fs-1_fmin-0_fmax-0-5,NaN
171,pyspi,ppc_multitaper_mean_fs-1_fmin-0_fmax-0-5,NaN


In [ ]:
import gc

def process_density_combo(combo_idx):
    row     = combos.iloc[combo_idx]
    method  = row["method"]
    func    = row["func"]
    lag     = row["lag"]
    lag_val = None if pd.isna(lag) else int(lag)

    avg_df = load_avg_mats_and_impose_sparsity(
        PARQUET_PATH, method=method, func=func,
        sub_list=SUB_LIST, sparsity=SPARSITY,
        symmetry_lookup=SYMMETRY_LOOKUP, lag=lag_val or 0,
    )

    valid = avg_df[avg_df["Matrix"].notna() & (avg_df["Count"] > 0)]
    if valid.empty:
        return pd.DataFrame()

    rows = []
    for sub, row_data in valid.iterrows():
        try:
            metrics = compute_all_density(
                row_data["Matrix"],
                sparsity=SPARSITY,
                tsp_nodes=TSP_NODES,
                small_world_kwargs=SMALL_WORLD_KWARGS,
                rich_club_kwargs=RICH_CLUB_KWARGS,
            )
        except Exception as exc:
            print(f"  [WARN] sub={sub} {method}/{func}/lag={lag}: {exc}")
            metrics = {}
        rows.append({"sub": sub, "method": method, "func": func, "lag": lag, **metrics})

    print(f"{method}/{func}/lag={lag} — {len(rows)} subjects")
    return pd.DataFrame(rows)


all_dfs = []
for i in range(len(combos)):
    df = process_density_combo(i)
    if not df.empty:
        all_dfs.append(df)
    # force mem free
    gc.collect()

density_df = pd.concat(all_dfs, ignore_index=True)
density_df.to_parquet(OUT_DIR / "info_density.parquet", index=False)
print(f"\nDone — {len(density_df)} rows saved")